In [22]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException  # Import ElementClickInterceptedException
import time

## Scrap The Artificial Plants

In [23]:
# URL to scrape
url = 'https://stocktrack.ca/?s=ikea&search=artificial%20plants'

# Initialize the Chrome WebDriver
driver = webdriver.Chrome()

# Navigate to the provided URL
driver.get(url)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 50)  # Set a timeout of 40 seconds
    # Locate the iframe by its tag name and switch to it
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")


start_dhx_f_id = 1
scraped_products = []
while True:
    try:
        # Wait for the products to load on the current page
        WebDriverWait(driver, 70).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
        
#         # Initialize a variable to track whether there are elements with dhx_f_id on the current page
        elements_found = False

        # Scrape the products on the current page
        for i in range(start_dhx_f_id, start_dhx_f_id + 5):
            div_xpath = f'//div[@dhx_f_id="{i}"]'

            for _ in range(3):
                try:
                    div_to_click = driver.find_element(By.XPATH, div_xpath)
                    if div_to_click:
                        div_to_click.click()
                    else:
                        break
                    div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

                    # Extract product information from the div element
                    image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
                    product_name = div_element.find_element(By.TAG_NAME, 'a').text
                    product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

                    # Split the text by line breaks to extract individual pieces of information
                    # Extract the text content from the parent element
                    product_info = div_element.text

                    # Split the text into lines and extract the relevant information
                    lines = product_info.split('\n')

                    # Initialize variables to store extracted information
                    sku = None
                    size = None
                    price = None

                    # Iterate through the lines to find relevant information
                    for line in lines:
                        if line.startswith("SKU:"):
                            sku = line.replace("SKU:", "").strip()
                        elif "cm" in line:
                            size = line.strip()
                        elif line.startswith("Price:"):
                            price = line.replace("Price:", "").strip()
                            
                    # Split the price into old and new price if applicable
                    if " " in price:
                        prices = price.split()
                        old_price = prices[0]  # First part is old price
                        new_price = prices[1]  # Second part is new price
                    else:
                        old_price = price
                        new_price = 'N/A'                        
                    # Print the extracted information
                    print(i)
                    print("image:", image)
                    print("Product Name :", product_name)
                    print("Product Link :", product_link)
                    print("SKU:", sku)
                    print("Size:", size)
                    print("Old Price:", old_price)
                    print("New Price:", new_price)
                    try:
                        stock_number_element = WebDriverWait(driver, 25).until(
                        EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
                        )
                        stock_number_coq = stock_number_element.text
                        print("Coquitlam Store Stock Number:", stock_number_coq)
                        
                        coq_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[2]")
                        stock_prob_coq = coq_prob_element.text
                        print("Coquitlam Store Stock Probability:", stock_prob_coq)
                                         
                        rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
                        stock_number_rich = rich_element.text
                        print("Richmond Store Stock Number:", stock_number_rich)
                        
                        rich_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[2]")
                        stock_prob_rich = rich_prob_element.text
                        print("Richmond Store Stock Probability:", stock_prob_rich)                        
                    except TimeoutException:
                        print("Timed out waiting for the Coquitlam stock number to load")
                    except NoSuchElementException:
                        print("Coquitlam stock number element not found.")       
                    print()
                    product_data = {
                    'image_url': image,
                    'product_name': product_name,
                    'product_link': product_link,
                    'product_size' : size,
                    'product_sku': sku,
                    'product_price_old': old_price,
                    'product_price_new' : new_price,
                    'stock_probability_coquitlam' : stock_prob_coq,   
                    'stock_number_coquitlam' : stock_number_coq,
                    'stock_probability_richmond' : stock_prob_rich,
                    'stock_number_richmond' : stock_number_rich
                    }
                    # Append the product data to the list
                    scraped_products.append(product_data)
                    # Set elements_found to True since elements with dhx_f_id were found
                    elements_found = True

                    break  # Exit the loop if the click is successful
                except StaleElementReferenceException:
                    continue  # Retry if a StaleElementReferenceException occurs
                except ElementClickInterceptedException:
                    print("Element click intercepted. Trying again.")
        
        # If no elements with dhx_f_id were found on the current page, break out of the loop
        if not elements_found:
            break
        print()
        
        # Check if the "Next" button is clickable
        next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")
        if not next_page_link.is_enabled():
            break  # Break out of the loop if the "Next" button is not clickable

        # Move to the next page by clicking the 'next page' link
        ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()
        start_dhx_f_id += 5
    except TimeoutException:
        print("Timed out waiting for products to load.")
    
    except NoSuchElementException:
            print(f"Element with dhx_f_id='{i}' not found. Exiting loop.")
            break  # Exit the loop if the element is not found



# Close the WebDriver
# Print the scraped product data

time.sleep(10)
driver.quit()





1
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/
SKU: 10467804
Size: 23 cm (9 ")
Old Price: $69.99
New Price: N/A
Coquitlam Store Stock Number: 16
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 9
Richmond Store Stock Probability: HIGH_IN_STOCK

2
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac__1184665_pe898020_s5.jpg
Product Name : FEJKA, Artificial plant with wall holder
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac-30548625/
SKU: 30548625
Size: None
Old Price: $6.99
New Price: N/A
Coquitlam Store Stock Number: 96
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 142

Coquitlam Store Stock Number: 10
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 10
Richmond Store Stock Probability: HIGH_IN_STOCK

17
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-string-of-bananas-hanging__1034086_pe840197_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-string-of-bananas-hanging-20508408/
SKU: 20508408
Size: 9 cm (3 ½ ")
Old Price: $1.99
New Price: N/A
Coquitlam Store Stock Number: 231
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 495
Richmond Store Stock Probability: HIGH_IN_STOCK

18
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-leaf-eucalyptus-green__0638910_pe699263_s5.jpg
Product Name : SMYCKA, Artificial leaf
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-leaf-eucalyptus-green-80335773/
SKU: 80335773
Size: 65 cm (25 ½ ")
O

33
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-fern__0898090_pe782562_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-fern-20468450/
SKU: 20468450
Size: 15 cm (6 ")
Old Price: $16.99
New Price: N/A
Coquitlam Store Stock Number: 58
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.

34
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-maple__1188014_pe899329_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-maple-60559957/
SKU: 60559957
Size: 12 cm (4 ¾ ")
Old Price: $11.99
New Price: N/A
Coquitlam Store Stock Number: 22
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.

35
image: https://www.ikea.com/ca/en/images/products/fejka-a

Coquitlam Store Stock Number: 23
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 15
Richmond Store Stock Probability: HIGH_IN_STOCK

50
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-baby-s-tears__0614213_pe686800_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-baby-s-tears-00532777/
SKU: 00532777
Size: 9 cm (3 ½ ")
Old Price: $6.99
New Price: N/A
Coquitlam Store Stock Number: 417
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Coquitlam stock number element not found.


51
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-indoor-outdoor-star__1007315_pe826019_s5.jpg
Product Name : SMYCKA, Artificial bouquet
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-bouquet-indoor-outdoor-star-40496541/
SKU: 40496541
Size: 40 cm (15 ¾ ")
Old Price: $2.99
New Price: N/A
Coquitlam Store St

Coquitlam Store Stock Number: 0
Coquitlam Store Stock Probability: OUT_OF_STOCK
Richmond Store Stock Number: 19
Richmond Store Stock Probability: HIGH_IN_STOCK

67
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-spray-indoor-outdoor-thistle-purple__1188135_pe899369_s5.jpg
Product Name : SMYCKA, Artificial spray
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-spray-indoor-outdoor-thistle-purple-20560123/
SKU: 20560123
Size: 45 cm (17 ¾ ")
Old Price: $2.99
New Price: N/A
Timed out waiting for the Coquitlam stock number to load

68
image: https://www.ikea.com/ca/en/images/products/vinterfint-artificial-potted-plant-indoor-outdoor-arrangement-red__1207942_pe908242_s5.jpg
Product Name : VINTERFINT, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/vinterfint-artificial-potted-plant-indoor-outdoor-arrangement-red-90562147/
SKU: 90562147
Size: 12 cm (4 ¾ ")
Old Price: $11.99
New Price: N/A
Timed out waiting for the Coquitlam stock number to 

84
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-mosaic-plant-hanging__1248044_pe922954_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-mosaic-plant-hanging-40571677/
SKU: 40571677
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 63
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 14
Richmond Store Stock Probability: HIGH_IN_STOCK

85
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-arrangement__1248031_pe922944_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-arrangement-80571675/
SKU: 80571675
Size: 9 cm (3 ½ ")
Old Price: $5.99
New Price: N/A
Coquitlam Store Stock Number: 8
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Num

Coquitlam Store Stock Number: 4
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 1
Richmond Store Stock Probability: LOW_IN_STOCK


101
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-tulip-pink__1248037_pe922950_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-tulip-pink-60571681/
SKU: 60571681
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 39
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 37
Richmond Store Stock Probability: HIGH_IN_STOCK

102
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-camellia-red__1248066_pe922970_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-camellia-red-50571790/
SKU: 50571790
Size: 28 cm (11 ")
Old Pri

In [24]:
for product in scraped_products:
    print(product)
    print() 

{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-bamboo__0748884_pe745273_s5.jpg', 'product_name': 'FEJKA, Artificial potted plant', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-bamboo-10467804/', 'product_size': '23 cm (9 ")', 'product_sku': '10467804', 'product_price_old': '$69.99', 'product_price_new': 'N/A', 'stock_probability_coquitlam': 'HIGH_IN_STOCK', 'stock_number_coquitlam': '16', 'stock_probability_richmond': 'HIGH_IN_STOCK', 'stock_number_richmond': '9'}

{'image_url': 'https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac__1184665_pe898020_s5.jpg', 'product_name': 'FEJKA, Artificial plant with wall holder', 'product_link': 'https://www.ikea.com/ca/en/p/fejka-artificial-plant-with-wall-holder-indoor-outdoor-green-lilac-30548625/', 'product_size': None, 'product_sku': '30548625', 'product_price_old': '$6.99', 'product_pri

In [25]:
import csv
import datetime

# Get the current date
current_date = datetime.datetime.now()

# Format the date as YYYY_MM_DD
formatted_date = current_date.strftime("%Y_%m_%d")
formatted_date2 = current_date.strftime("%Y/%m/%d")

# Create a dynamic filename based on the current date
csv_file_path = f"products_artifical_plants_{formatted_date}.csv"

# Create or open the CSV file for writing
with open(csv_file_path, mode='w', newline='') as csv_file:
    # Define the CSV headers (column names)
    fieldnames = [
        'current_date',
        'image_url',
        'product_name',
        'product_link',
        'product_size',
        'product_sku',
        'product_price_old',
        'product_price_new',
        'stock_probability_coquitlam',
        'stock_number_coquitlam',
        'stock_probability_richmond',
        'stock_number_richmond'
    ]

    # Create a CSV writer
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Write the header row to the CSV file
    csv_writer.writeheader()

    # Iterate through the scraped_products list and write each product's data
    for product in scraped_products:
        product['current_date'] = formatted_date2
        csv_writer.writerow(product)

print(f"CSV file has been created : '{csv_file_path}'.")

CSV file has been created : 'products_artifical_plants_2024_01_31.csv'.


### Scrap The Real Plants

In [26]:
# URL to scrape
url = 'https://stocktrack.ca/?s=ikea&search=real%20plants'

# Initialize the Chrome WebDriver
driver = webdriver.Chrome()

# Navigate to the provided URL
driver.get(url)

# Wait for the iframe to load and switch to it
try:
    wait = WebDriverWait(driver, 50)  # Set a timeout of 40 seconds
    # Locate the iframe by its tag name and switch to it
    iframe = wait.until(EC.presence_of_element_located((By.TAG_NAME, 'iframe')))
    driver.switch_to.frame(iframe)
except TimeoutException:
    print("No iframe found or timed out waiting for iframe to load.")
except NoSuchElementException:
    print("No iframe element found.")


start_dhx_f_id = 1
scraped_products = []
while True:
    try:
        # Wait for the products to load on the current page
        WebDriverWait(driver, 60).until(EC.presence_of_all_elements_located((By.CLASS_NAME, "dhx_list_item")))
        
#         # Initialize a variable to track whether there are elements with dhx_f_id on the current page
        elements_found = False

        # Scrape the products on the current page
        for i in range(start_dhx_f_id, start_dhx_f_id + 5):
            div_xpath = f'//div[@dhx_f_id="{i}"]'

            for _ in range(3):
                try:
                    div_to_click = driver.find_element(By.XPATH, div_xpath)
                    if div_to_click:
                        div_to_click.click()
                    else:
                        break
                    div_element = driver.find_element(By.XPATH, f'//div[@dhx_f_id="{i}"]')

                    # Extract product information from the div element
                    image = div_element.find_element(By.TAG_NAME, 'img').get_attribute('src')
                    product_name = div_element.find_element(By.TAG_NAME, 'a').text
                    product_link = div_element.find_element(By.TAG_NAME, 'a').get_attribute('href')

                    # Split the text by line breaks to extract individual pieces of information
                    # Extract the text content from the parent element
                    product_info = div_element.text

                    # Split the text into lines and extract the relevant information
                    lines = product_info.split('\n')

                    # Initialize variables to store extracted information
                    sku = None
                    size = None
                    price = None

                    # Iterate through the lines to find relevant information
                    for line in lines:
                        if line.startswith("SKU:"):
                            sku = line.replace("SKU:", "").strip()
                        elif "cm" in line:
                            size = line.strip()
                        elif line.startswith("Price:"):
                            price = line.replace("Price:", "").strip()
                            
                    # Split the price into old and new price if applicable
                    if " " in price:
                        prices = price.split()
                        old_price = prices[0]  # First part is old price
                        new_price = prices[1]  # Second part is new price
                    else:
                        old_price = price
                        new_price = 'N/A'                        
                    # Print the extracted information
                    print(i)
                    print("image:", image)
                    print("Product Name :", product_name)
                    print("Product Link :", product_link)
                    print("SKU:", sku)
                    print("Size:", size)
                    print("Old Price:", old_price)
                    print("New Price:", new_price)
                    try:
                        stock_number_element = WebDriverWait(driver, 25).until(
                        EC.presence_of_element_located((By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[3]"))
                        )
                        stock_number_coq = stock_number_element.text
                        print("Coquitlam Store Stock Number:", stock_number_coq)
                        
                        coq_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Coquitlam')]/following-sibling::td[2]")
                        stock_prob_coq = coq_prob_element.text
                        print("Coquitlam Store Stock Probability:", stock_prob_coq)
                                         
                        rich_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[3]")
                        stock_number_rich = rich_element.text
                        print("Richmond Store Stock Number:", stock_number_rich)
                        
                        rich_prob_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Richmond')]/following-sibling::td[2]")
                        stock_prob_rich = rich_prob_element.text
                        print("Richmond Store Stock Probability:", stock_prob_rich)    
                    except TimeoutException:
                        print("Timed out waiting for the Coquitlam stock number to load")
                    except NoSuchElementException:
                        print("Coquitlam stock number element not found.")       
                    print()
                    product_data = {
                    'image_url': image,
                    'product_name': product_name,
                    'product_link': product_link,
                    'product_size' : size,
                    'product_sku': sku,
                    'product_price_old': old_price,
                    'product_price_new' : new_price,
                    'stock_probability_coquitlam' : stock_prob_coq,   
                    'stock_number_coquitlam' : stock_number_coq,
                    'stock_probability_richmond' : stock_prob_rich,
                    'stock_number_richmond' : stock_number_rich
                    }
                    # Append the product data to the list
                    scraped_products.append(product_data)
                    # Set elements_found to True since elements with dhx_f_id were found
                    elements_found = True

                    break  # Exit the loop if the click is successful
                except StaleElementReferenceException:
                    continue  # Retry if a StaleElementReferenceException occurs
                except ElementClickInterceptedException:
                    print("Element click intercepted. Trying again.")
        
        # If no elements with dhx_f_id were found on the current page, break out of the loop
        if not elements_found:
            break
        print()
        
        # Check if the "Next" button is clickable
        next_page_link = driver.find_element(By.XPATH, "//div[@dhx_p_id='next']")
        if not next_page_link.is_enabled():
            break  # Break out of the loop if the "Next" button is not clickable

        # Move to the next page by clicking the 'next page' link
        ActionChains(driver).move_to_element(next_page_link).click(next_page_link).perform()
        start_dhx_f_id += 5
    except TimeoutException:
        print("Timed out waiting for products to load.")
    
    except NoSuchElementException:
            print(f"Element with dhx_f_id='{i}' not found. Exiting loop.")
            break  # Exit the loop if the element is not found



# Close the WebDriver
# Print the scraped product data

time.sleep(10)
driver.quit()





1
image: https://www.ikea.com/ca/en/images/products/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage__67453_pe181294_s5.jpg
Product Name : HIMALAYAMIX, Potted plant
Product Link : https://www.ikea.com/ca/en/p/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage-20197227/
SKU: 20197227
Size: 10 cm (4 ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 77
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 50
Richmond Store Stock Probability: HIGH_IN_STOCK

2
image: https://www.ikea.com/ca/en/images/products/sansevieria-trifasciata-potted-plant-mother-in-laws-tongue__0237399_pe376787_s5.jpg
Product Name : SANSEVIERIA TRIFASCIATA, Potted plant
Product Link : https://www.ikea.com/ca/en/p/sansevieria-trifasciata-potted-plant-mother-in-laws-tongue-90296969/
SKU: 90296969
Size: 15 cm (6 ")
Old Price: $9.99
New Price: N/A
Coquitlam Store Stock Number: 72
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store St

Coquitlam Store Stock Number: 161
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 180
Richmond Store Stock Probability: HIGH_IN_STOCK

18
image: https://www.ikea.com/ca/en/images/products/cactaceae-potted-plant-assorted-species-plants-cactus__0492123_pe625477_s5.jpg
Product Name : CACTACEAE, Potted plant
Product Link : https://www.ikea.com/ca/en/p/cactaceae-potted-plant-assorted-species-plants-cactus-90120060/
SKU: 90120060
Size: 10 cm (4 ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 40
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 19
Richmond Store Stock Probability: HIGH_IN_STOCK

19
image: https://www.ikea.com/ca/en/images/products/spathiphyllum-potted-plant-peace-lily__0653998_pe708227_s5.jpg
Product Name : SPATHIPHYLLUM, Potted plant
Product Link : https://www.ikea.com/ca/en/p/spathiphyllum-potted-plant-peace-lily-00197902/
SKU: 00197902
Size: 15 cm (6 ")
Old Price: $14.99
New Price: N/A
Coquitlam Sto

Coquitlam Store Stock Number: 13
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 10
Richmond Store Stock Probability: HIGH_IN_STOCK

35
image: https://www.ikea.com/ca/en/images/products/anthurium-potted-plant-flamingo-plant-assorted-colors__0657278_pe709725_s5.jpg
Product Name : ANTHURIUM, Potted plant
Product Link : https://www.ikea.com/ca/en/p/anthurium-potted-plant-flamingo-plant-assorted-colors-00399108/
SKU: 00399108
Size: 10 cm (4 ")
Old Price: $12.99
New Price: N/A
Coquitlam Store Stock Number: 0
Coquitlam Store Stock Probability: OUT_OF_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK


36
image: https://www.ikea.com/ca/en/images/products/poinsettia-potted-plant-poinsettia__1189322_pe899783_s5.jpg
Product Name : POINSETTIA, Potted plant
Product Link : https://www.ikea.com/ca/en/p/poinsettia-potted-plant-poinsettia-40562456/
SKU: 40562456
Size: 10 cm (4 ")
Old Price: $6.99
New Price: N/A
Timed out waiting for the C

52
image: https://www.ikea.com/ca/en/images/products/cyclamen-potted-plant-alpine-violet__0543449_pe656714_s5.jpg
Product Name : CYCLAMEN, Potted plant
Product Link : https://www.ikea.com/ca/en/p/cyclamen-potted-plant-alpine-violet-50563648/
SKU: 50563648
Size: 10 cm (4 ")
Old Price: $5.99
New Price: N/A
Timed out waiting for the Coquitlam stock number to load

53
image: https://www.ikea.com/ca/en/images/products/vinterfint-artificial-potted-plant-indoor-outdoor-arrangement-red__1207942_pe908242_s5.jpg
Product Name : VINTERFINT, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/vinterfint-artificial-potted-plant-indoor-outdoor-arrangement-red-90562147/
SKU: 90562147
Size: 12 cm (4 ¾ ")
Old Price: $11.99
New Price: N/A
Timed out waiting for the Coquitlam stock number to load

54
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-plant-wall-mounted-indoor-outdoor-moss-green__1183264_pe897461_s5.jpg
Product Name : FEJKA, Artificial plant
Product Link : ht

Coquitlam Store Stock Number: 11
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 16
Richmond Store Stock Probability: HIGH_IN_STOCK

69
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-rubber-plant__1154600_pe886223_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-rubber-plant-60548313/
SKU: 60548313
Size: 23 cm (9 ")
Old Price: $79.99
New Price: N/A
Coquitlam Store Stock Number: 4
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 8
Richmond Store Stock Probability: HIGH_IN_STOCK

70
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-maple__1188014_pe899329_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-maple-60559957/
SKU: 60559957
Size: 12 cm (4 ¾ ")
Old

...

101
image: https://www.ikea.com/ca/en/images/products/tulipa-potted-plant-tulip__0935994_pe793046_s5.jpg
Product Name : TULIPA, Potted plant
Product Link : https://www.ikea.com/ca/en/p/tulipa-potted-plant-tulip-00569983/
SKU: 00569983
Size: 10 cm (4 ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 25
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 20
Richmond Store Stock Probability: HIGH_IN_STOCK

102
image: https://www.ikea.com/ca/en/images/products/aechmea-potted-plant-urn-plant__0485365_pe621494_s5.jpg
Product Name : AECHMEA, Potted plant
Product Link : https://www.ikea.com/ca/en/p/aechmea-potted-plant-urn-plant-00569997/
SKU: 00569997
Size: 15 cm (6 ")
Old Price: $22.99
New Price: N/A
Coquitlam Store Stock Number: 0
Coquitlam Store Stock Probability: OUT_OF_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK

103
image: https://www.ikea.com/ca/en/images/products/narcissus-potted-plant-narcissus__0935993_

118
image: https://www.ikea.com/ca/en/images/products/sansevieria-plant-with-pot-snake-plant-assorted-colors__1169057_pe892338_s5.jpg
Product Name : SANSEVIERIA, Plant with pot
Product Link : https://www.ikea.com/ca/en/p/sansevieria-plant-with-pot-snake-plant-assorted-colors-60445796/
SKU: 60445796
Size: 6 cm (2 ¼ ")
Old Price: $6.99
New Price: N/A
Coquitlam Store Stock Number: 64
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 32
Richmond Store Stock Probability: HIGH_IN_STOCK

119
image: https://www.ikea.com/ca/en/images/products/fejka-artificial-potted-plant-indoor-outdoor-rose-pink__0614177_pe686803_s5.jpg
Product Name : FEJKA, Artificial potted plant
Product Link : https://www.ikea.com/ca/en/p/fejka-artificial-potted-plant-indoor-outdoor-rose-pink-00395313/
SKU: 00395313
Size: 9 cm (3 ½ ")
Old Price: $4.99
New Price: N/A
Coquitlam Store Stock Number: 25
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 70
Richmond Store Sto

135
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-indoor-outdoor-bouguet-multicolor-sweet-pea__1248058_pe922962_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-indoor-outdoor-bouguet-multicolor-sweet-pea-90571806/
SKU: 90571806
Size: 33 cm (13 ")
Old Price: $6.99
New Price: N/A
Coquitlam Store Stock Number: 22
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 17
Richmond Store Stock Probability: HIGH_IN_STOCK


136
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-indoor-outdoor-dahlia__1248045_pe922956_s5.jpg
Product Name : SMYCKA, Artificial bouquet
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-bouquet-indoor-outdoor-dahlia-90571825/
SKU: 90571825
Size: 55 cm (22 ")
Old Price: $16.99
New Price: N/A
Coquitlam Store Stock Number: 5
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 6
Richmond S

Coquitlam Store Stock Number: 23
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 38
Richmond Store Stock Probability: HIGH_IN_STOCK

153
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-spray-indoor-outdoor-dogwood-white__1188154_pe899381_s5.jpg
Product Name : SMYCKA, Artificial spray
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-spray-indoor-outdoor-dogwood-white-50560145/
SKU: 50560145
Size: 100 cm (39 ¼ ")
Old Price: $9.99
New Price: N/A
Coquitlam Store Stock Number: 8
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 0
Richmond Store Stock Probability: OUT_OF_STOCK

154
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-flower-queen-ann-stem-pink__1188158_pe899385_s5.jpg
Product Name : SMYCKA, Artificial flower
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-flower-queen-ann-stem-pink-50562738/
SKU: 50562738
Size: 65 cm (25 ½ ")
Old Price: $3.99
New Price: N/A

Coquitlam Store Stock Number: 319
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 180
Richmond Store Stock Probability: HIGH_IN_STOCK

170
image: https://www.ikea.com/ca/en/images/products/smycka-artificial-bouquet-indoor-outdoor-green__0817337_pe773968_s5.jpg
Product Name : SMYCKA, Artificial bouquet
Product Link : https://www.ikea.com/ca/en/p/smycka-artificial-bouquet-indoor-outdoor-green-40461136/
SKU: 40461136
Size: 31 cm (12 ¼ ")
Old Price: $2.99
New Price: N/A
Coquitlam Store Stock Number: 85
Coquitlam Store Stock Probability: HIGH_IN_STOCK
Richmond Store Stock Number: 74
Richmond Store Stock Probability: HIGH_IN_STOCK


171
image: https://www.ikea.com/ca/en/images/products/konstfull-vase-green__1030424_pe836277_s5.jpg
Product Name : KONSTFULL, Vase
Product Link : https://www.ikea.com/ca/en/p/konstfull-vase-green-30511962/
SKU: 30511962
Size: 19 cm (7 ½ ")
Old Price: $19.99
New Price: N/A
Coquitlam Store Stock Number: 24
Coquitlam Store Stock Probabi

In [27]:
for product in scraped_products:
    print(product)
    print() 

{'image_url': 'https://www.ikea.com/ca/en/images/products/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage__67453_pe181294_s5.jpg', 'product_name': 'HIMALAYAMIX, Potted plant', 'product_link': 'https://www.ikea.com/ca/en/p/himalayamix-potted-plant-assorted-species-plants-plants-with-foliage-20197227/', 'product_size': '10 cm (4 ")', 'product_sku': '20197227', 'product_price_old': '$4.99', 'product_price_new': 'N/A', 'stock_probability_coquitlam': 'HIGH_IN_STOCK', 'stock_number_coquitlam': '77', 'stock_probability_richmond': 'HIGH_IN_STOCK', 'stock_number_richmond': '50'}

{'image_url': 'https://www.ikea.com/ca/en/images/products/sansevieria-trifasciata-potted-plant-mother-in-laws-tongue__0237399_pe376787_s5.jpg', 'product_name': 'SANSEVIERIA TRIFASCIATA, Potted plant', 'product_link': 'https://www.ikea.com/ca/en/p/sansevieria-trifasciata-potted-plant-mother-in-laws-tongue-90296969/', 'product_size': '15 cm (6 ")', 'product_sku': '90296969', 'product_price_old': '$9.

In [28]:
# Get the current date
current_date = datetime.datetime.now()

# Format the date as YYYY_MM_DD
formatted_date = current_date.strftime("%Y_%m_%d")
formatted_date2 = current_date.strftime("%Y/%m/%d")

# Create a dynamic filename based on the current date
csv_file_path = f"products_real_plants_{formatted_date}.csv"

# Create or open the CSV file for writing
with open(csv_file_path, mode='w', newline='') as csv_file:
    # Define the CSV headers (column names)
    fieldnames = [
        'current_date',
        'image_url',
        'product_name',
        'product_link',
        'product_size',
        'product_sku',
        'product_price_old',
        'product_price_new',
        'stock_probability_coquitlam',
        'stock_number_coquitlam',
        'stock_probability_richmond',
        'stock_number_richmond'
    ]

    # Create a CSV writer
    csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

    # Write the header row to the CSV file
    csv_writer.writeheader()

    # Iterate through the scraped_products list and write each product's data
    for product in scraped_products:
        product['current_date'] = formatted_date2
        csv_writer.writerow(product)

print(f"CSV file has been created : '{csv_file_path}'.")

CSV file has been created : 'products_real_plants_2024_01_31.csv'.
